# Lesson 4.1 · From the line to the probability

In module 1 you predicted a number with a line. Now the target is a **class** (default yes/no) and you want a
**probability**. Logistic regression is the linear model that answers this question: it is the baseline of almost every
classification problem, and, like linear regression, it gives you interpretable coefficients besides the prediction.

**What you will learn**

- Why linear regression is not suitable for a probability, and what the sigmoid adds.
- To read the coefficients as **log-odds** and **odds ratios**.
- What `fit` minimizes: the **log-loss**, and why the squared error is not used.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]                          # how much the loan weighs on the income
print(df[["ratio", "age", "defaulted"]].describe().round(2).loc[["mean", "std", "min", "max"]])
print("default rate:", round(df["defaulted"].mean(), 3))

## Why linear regression is not enough

Let's try the method from module 1 on the 0/1 target using a single feature, the amount/income ratio.

In [ ]:
x = df[["ratio"]].to_numpy()
y = df["defaulted"].to_numpy()
lin = LinearRegression().fit(x, y)
log = LogisticRegression(C=1e6, max_iter=1000).fit(x, y)                 # huge C = almost no regularization

grid = np.linspace(0, x.max(), 300).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(x, y + np.random.default_rng(0).uniform(-0.03, 0.03, len(y)), s=6, alpha=0.3, color="gray")
ax.plot(grid, lin.predict(grid), label="line (linear regression)")
ax.plot(grid, log.predict_proba(grid)[:, 1], label="sigmoid (logistic regression)")
ax.axhline(0, color="k", lw=0.5); ax.axhline(1, color="k", lw=0.5)
ax.set_xlabel("amount / income"); ax.set_ylabel("P(default)"); ax.legend(); plt.show()
print("the line predicts values outside [0, 1]:", (lin.predict(x) < 0).sum(), "rows below 0,", (lin.predict(x) > 1).sum(), "above 1")

The line produces "probabilities" outside [0, 1] (here 11 loans get more than 1) and has no probabilistic interpretation. Logistic regression applies to the linear model a
function that squashes every real number into (0, 1): the **sigmoid** $\sigma(z) = 1/(1+e^{-z})$.

$$P(y=1\mid x) = \sigma(w_0 + w_1 x_1 + \dots + w_p x_p)$$

**Intuition.** The model stays linear, but on a different scale: $z = w_0 + \mathbf{w}^\top \mathbf{x}$ is the **log-odds** (the logarithm
of $p/(1-p)$). Every extra unit of $x_j$ adds $w_j$ to the log-odds, that is **multiplies the odds by $e^{w_j}$**.

## Coefficients as odds ratios

Let's standardize the features (scaling has a precise reason, you will see it in lesson 4.2) and read the coefficients.

In [ ]:
cols = ["ratio", "age", "job_tenure"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.25, stratify=df["defaulted"], random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

w = model[-1].coef_[0]
table = pd.DataFrame({"coef (log-odds per 1 std dev)": w, "odds ratio": np.exp(w)}, index=cols)
print(table.round(3))
print("intercept:", model[-1].intercept_.round(3), "-> baseline probability of an average person:", round(1 / (1 + np.exp(-model[-1].intercept_[0])), 3))
print("test AUC:", round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3))

An odds ratio of 3.3 for `ratio` reads: "all other features being equal, an increase of one standard deviation of the
amount/income ratio multiplies the default **odds** by 3.3". Not the probability: the odds. With small probabilities
the two are close, but the difference grows with p.

> **Common mistake.** Reading a coefficient as the "pure" effect of the feature. The coefficient holds **all else being equal**: if
> two features are correlated (here `age` and `job_tenure`), the effect is split between the two in an unstable way. You will see it in exercise D1.

## What `fit` minimizes: the log-loss

For every example with predicted probability $\hat p$ and true class $y$, the loss is $-[y\log \hat p + (1-y)\log(1-\hat p)]$: small if
you assign a high probability to the right class, enormous if you are sure and wrong. The mean over the training set is the **log-loss** (or cross
entropy). The squared error is not used because with the sigmoid it becomes nonconvex with vanishing gradients; the log-loss is
convex: a single minimum, found by an iterative solver (`lbfgs` by default).

In [ ]:
p = model.predict_proba(X_te)[:, 1]
manual = -np.mean(y_te * np.log(p) + (1 - y_te) * np.log(1 - p))
print("manual log-loss:", round(manual, 4), "| sklearn:", round(log_loss(y_te, p), 4))
print("log-loss of someone who always predicts the mean rate:", round(log_loss(y_te, np.full(len(y_te), y_tr.mean())), 4))

The second number is the baseline: a model that uses no feature. The log-loss of the model must be **below** that figure.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>What is the difference between `predict` and `predict_proba`?</summary>

`predict_proba` returns the probabilities (one column per class, in `classes_` order). `predict` applies the 0.5 threshold and returns the labels. If you need another threshold (lesson 4.3) you must start from `predict_proba`.

</details>

<details><summary>If a coefficient is 0.7, by how much does the probability change with +1 in the feature?</summary>

There is no single answer: the log-odds change by 0.7 (the odds are multiplied by e^0.7 ≈ 2.01), but the change in probability depends on the starting point. Near p = 0.5 the slope is maximal (about w/4); near 0 or 1 the sigmoid is almost flat.

</details>